In [1]:
!git clone https://github.com/hung20gg/grade_func.git
!cd grade_func && git pull

fatal: destination path 'grade_func' already exists and is not an empty directory.
Already up to date.


In [2]:
from grade_func.grader import test_func_from_file

### Ex1: Write a function, using regex to validate the datetime format (1.5 point)
Given a datetime as a string, check does the datetime satisfy the condition DD-MM-YYYY or DD/MM/YYYY. Return True or False

In [3]:
import re

In [4]:
def validate_datetime(string : str) -> bool:
    pattern = re.compile(r'^(0[1-9]|[12][0-9]|3[0-1])[-/](0[1-9]|1[0-2])[-/](\d{4})$')
    match = pattern.match(string)
    
    if not match:
        return False
    
    day, month, year = int(match.group(1)), int(match.group(2)), int(match.group(3))
    
    if not (1 <= month <= 12):
        return False
    
    if month == 2:
        if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
            return 1 <= day <= 29
        else:
            return 1 <= day <= 28
    elif month in {4, 6, 9, 11}:
        return 1 <= day <= 30
    else:
        return 1 <= day <= 31

In [5]:
validate_datetime("29/02/2024")


True

### Ex2: Sum of all element (1.5 point)
Find the sum of all numerical element passed into the function

In [6]:
def sum_(*args) -> int:
    sum = 0
    for element in args:
        if isinstance(element, (int, float)):
            sum += element

    return sum


In [7]:
sum_(1, 2, 3, 4)

10

### Ex3: Dynamic function with **kwargs (2 point)

You are given 2 function:

$f(x_1, x_2, x_3, x_4) = x_1 + x_2^2 + x_3^2 + x_4$

$g(x_1, x_2) = x_1 + x_2$


You must write function f and g to satisfy the router below

In [19]:
def f(x1,x2,x3,x4):
    return x1 + x2**2 + x3**2 + x4

def g(x1,x2):
    return x1 + x2


# Evaluate func
def router(pick,x1,x2,x3,x4): # The testcase checker does not support **kwargs
    def your_func(pick, **kwargs):
        if pick == 'f':
            return f(**kwargs)
        elif pick == 'g':
            return g(kwargs['x1'], kwargs['x2'])
    
    return your_func(pick, x1=x1, x2=x2, x3=x3, x4=x4)

In [101]:
def f(x1,x2,x3,x4):
    return x1 + x2**2 + x3**2 + x4

def g(x1,x2):
    return x1 + x2


# Evaluate func
def router(pick,x1,x2,x3,x4): # The testcase checker does not support **kwargs
    def your_func(pick, **kwargs):
        if pick == 'f':
            return f(x1, x2, x3, x4)
        elif pick == 'g':
            return g(x1, x2)
    
    return your_func(pick, x1=x1, x2=x2, x3=x3, x4=x4)


In [25]:
print(router('f', 1, 2, 3, 4))


18


In [26]:
print(router('g', 1, 2, 3, 4))

3


### Ex4: Vectorize a sentence with Word2Vec (5 point)

For simplicity, we will ignore how Word2Vec was trained.

Word2Vec is a hashmap (dictionary) with keys are the words and values are the corresponding vectors (50-dim) to the word. To encode a sentence to a vector, you simply get the mean vector of words in the sentence.

Eg.
```
model = {
    "I": [0,1,2],
    "love":[1.5,2,0.3],
    "you":[0.5,1,2]
}
```
The encoding for sentence *I love you* is the mean value of `model["I"]`, `model["love"]`, "model["you"]`
```
vector = [2/3, 4/3, 4.3/3] 
```

In [50]:
import gensim.downloader as api
# Load the pre-trained Word2Vec model
model = api.load('glove-wiki-gigaword-50') # Here is your word2vec database

#### Encode a sentence (2 point)

In [67]:
import nltk
from gensim.models import Word2Vec
import gensim
from nltk.tokenize import word_tokenize

def clean_text(text : str) -> str: # Clean the text before processing
    return text.lower() 

def get_vector(text : str) -> list:
    text = clean_text(text)

    words = [word for word in word_tokenize(text) if word in model]
    if not words:
        return None
    vectors = [model[word] for word in words]
    return sum(vectors) / len(vectors)

def validate(text : str) -> bool:
    text = clean_text(text)
    words = word_tokenize(text)
    for word in words:
        if word not in model:
            return False
    return True


In [63]:
get_vector("I love you")


array([-0.00701397,  0.54196334, -0.19225435, -0.5246567 ,  0.7782766 ,
       -0.04985666, -0.31802335,  0.1746592 , -0.52432996,  0.4764192 ,
       -0.33454332,  0.93489003, -0.61863667, -0.164186  ,  1.1000067 ,
        0.33991334,  0.29203   ,  0.35769334,  0.07931166, -0.724163  ,
       -0.42256665,  0.87211996,  0.7086167 ,  0.45412335,  1.2277    ,
       -2.0613    , -1.3180667 ,  0.23561667,  1.2105001 , -1.2606801 ,
        3.3339665 ,  0.7460467 , -0.60946995,  0.23688667, -0.31138667,
       -0.179042  ,  0.17087667,  0.119286  ,  0.3511467 , -0.56632334,
        0.09226224, -0.03197267, -0.20612   ,  0.41710332,  0.168862  ,
        0.18619333,  0.08125467, -0.8010633 , -0.20057969,  0.78086996],
      dtype=float32)

In [66]:
validate("I love you")

True

### Calculate similarity of 2 sentence

In [72]:
from scipy.spatial.distance import cosine

In [75]:
def cosine_similarity(text1 : str, text2 : str) -> float:
    vector1 = get_vector(text1)
    vector2 = get_vector(text2)
    cosine_sim = 1 - cosine(vector1, vector2)
    return cosine_sim

In [98]:
cosine_similarity("I love you", "I love you")

0.9999999981511719

### Get top-k similarity (2 point)

Given a list of sentences (knowledge database) and a random sentence, return index of top-k similar sentence in the knowledge database

In [79]:
with open("knowledge.txt") as f:
    database = f.readlines()


In [99]:
def top_k_similar(text: str, k: int) -> list:
    text = clean_text(text)
    similarity = []
    for index, line in enumerate(database):
        line = line.strip()
        similarity.append((index, cosine_similarity(line, text)))

    similarity.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in similarity[:k]]


In [100]:
top_k_similar("I love you", 3)
# cau thu 10, cau thu 3, cau thu 13

[10, 3, 13]